In [ ]:
import pandas as pd

# === INPUT FILES ===
# Make sure these files are in the same folder as this script
mapping_file = "main_with_subs_only.xlsx"
indent_file  = "Monthly Indent.xlsx"

# === LOAD DATA ===
df_mapping = pd.read_excel(mapping_file)
df_indent   = pd.read_excel(indent_file)

# Rename columns to make the code easier to read (using your terminology)
df_mapping = df_mapping.rename(columns={
    'Main_Label': 'Child_Part',
    'Sub_Label':  'Switch_Part',
    'Main_Count': 'Qty_per_Switch',
    'Sub_Count':  'Historical_Total'   # kept but not used for this calculation
})

df_indent = df_indent.rename(columns={'Part number': 'Switch_Part'})

# List of months exactly as they appear in your "Monthly Indent.xlsx"
month_cols = ["Feb'26", "Mar'26", "Apr'26", "May'26", "Jun'26", "Jul'26"]

# === MERGE THE TWO TABLES ===
# We bring the monthly indent values into the mapping table
df_merged = pd.merge(
    df_mapping[['Child_Part', 'Switch_Part', 'Qty_per_Switch']],
    df_indent[['Switch_Part'] + month_cols],
    on='Switch_Part',
    how='left'  # keep all rows from mapping, even if no indent match
)

# === CALCULATE DAILY AND 2-DAYS REQUIREMENTS ===
for month in month_cols:
    # Create clean column names without apostrophe (Feb26 instead of Feb'26)
    daily_col   = f"Daily_{month.replace(\"'\", \"\")}"     # e.g. Daily_Feb26
    twodays_col = f"2Days_{month.replace(\"'\", \"\")}"     # e.g. 2Days_Feb26
    
    # Daily demand of the switch = monthly indent / 30
    df_merged[daily_col] = (df_merged[month] / 30.0).round(2)
    
    # Requirement for this child from this switch for 2 days
    df_merged[twodays_col] = (df_merged[daily_col] * df_merged['Qty_per_Switch'] * 2).round(2)

# ───────────────────────────────────────────────────────────────
# PART 1: TOTALS PER CHILD PART (one row per unique Child_Part)
# ───────────────────────────────────────────────────────────────
totals = df_merged.groupby('Child_Part', as_index=False).agg({
    f"Daily_{m.replace(\"'\", \"\")}": 'sum' for m in month_cols
})

# Add the 2Days columns for totals (just double the daily sum)
for month in month_cols:
    daily_col   = f"Daily_{month.replace(\"'\", \"\")}"
    twodays_col = f"2Days_{month.replace(\"'\", \"\")}"
    totals[twodays_col] = (totals[daily_col] * 2).round(2)

# Set nice column order
totals_cols = ['Child_Part'] + \
              [f"Daily_{m.replace(\"'\", \"\")}" for m in month_cols] + \
              [f"2Days_{m.replace(\"'\", \"\")}" for m in month_cols]

totals = totals[totals_cols]

# Save to file
totals_file = "Child_Totals_2Days_Per_Month.xlsx"
totals.to_excel(totals_file, index=False)
print(f"Totals file saved: {totals_file} ({len(totals)} rows)")

# ───────────────────────────────────────────────────────────────
# PART 2: DETAILED BREAKDOWN (one row per child + switch pair)
# ───────────────────────────────────────────────────────────────
detailed_cols = (
    ['Child_Part', 'Switch_Part', 'Qty_per_Switch'] +
    month_cols +
    [f"Daily_{m.replace(\"'\", \"\")}" for m in month_cols] +
    [f"2Days_{m.replace(\"'\", \"\")}" for m in month_cols]
)

detailed = df_merged[detailed_cols]

# Save to separate file
detailed_file = "Child_Detailed_Breakdown_2Days.xlsx"
detailed.to_excel(detailed_file, index=False)
print(f"Detailed file saved: {detailed_file} ({len(detailed)} rows)")

print("Done! Check the two new Excel files in your folder.")